# Análise Estrutura - Comunidades


## Tratamento da rede 

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import collections
import time
import warnings
import os

In [21]:
warnings.filterwarnings("ignore")

In [ ]:
EDGE_FILE = "com-lj.all.cmty.txt"

print("*" * 60)
print("ANÁLISE ESTRUTURAL (estimativas) - Grafo Grafo de comunidades")
print("*" * 60)

print("\n[1/9] Carregando grafo...")
t0 = time.time()

************************************************************
ANÁLISE ESTRUTURAL (estimativas) - Grafo Grafo de comunidades
************************************************************

[1/9] Carregando grafo...


In [23]:
G = nx.read_edgelist(
    EDGE_FILE, 
    comments="#",
    nodetype=int,
    create_using=nx.Graph(),
    data=False
)
print(f"  Tempo de carregamento: {time.time() - t0:.1f}s")


  Tempo de carregamento: 2.9s


##  2. Pre-processing

Removing auto loops

In [24]:
loops_before = nx.number_of_selfloops(G)
G.remove_edges_from(nx.selfloop_edges(G))
print(f"  Auto-loops removidos: {loops_before}")

if not nx.is_connected(G):
    Gcc = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    print(f"  Nós removidos ao extrair LCC: {G.number_of_nodes() - Gcc.number_of_nodes()}")
    G = Gcc
else:    
    print("  Grafo já é totalmente conexo.")


  Auto-loops removidos: 0
  Nós removidos ao extrair LCC: 174472


## 3. Report basic metrics

In [25]:
print("\n[3/9] Métricas básicas...")

n_nodes = G.number_of_edges()
n_edges  = G.number_of_nodes()
density = nx.density(G)

degrees = [d for _, d in G.degree()]
deg_min = min(degrees)
deg_max = max(degrees)
deg_avg = sum(degrees) / len(degrees)
deg_med = float(np.median(degrees))

print(f"  Vértices : {n_nodes:,}")
print(f"  Arestas  : {n_edges:,}")
print(f"  Densidade: {density:.8f}")
print(f"  Grau mín : {deg_min}")
print(f"  Grau máx : {deg_max}")
print(f"  Grau méd : {deg_avg:.4f}")
print(f"  Grau med : {deg_med:.1f}")



[3/9] Métricas básicas...
  Vértices : 427,701
  Arestas  : 303,526
  Densidade: 0.00000928
  Grau mín : 1
  Grau máx : 221
  Grau méd : 2.8182
  Grau med : 2.0


## 4. Componentes Conexas

In [26]:
components = list(nx.connected_components(G))
n_comp = len(components)
comp_sizes = sorted([len(c) for c in components], reverse=True)
print(f"  Número de componentes: {n_comp}")
print(f"  Tamanhos (top 5): {comp_sizes[:5]}")

  Número de componentes: 1
  Tamanhos (top 5): [303526]


## 5. Diâmetro e Raio 

In [27]:
print("\n[5/9] Diâmetro e Raio (estimativa por amostragem)...")

SAMPLE_K = 500 
rng = np.random.default_rng(42)
sample_nodes = list(rng.choice(list(G.nodes()), size=min(SAMPLE_K, G.number_of_nodes()), replace=False))

eccentricities_sample = {}
t0 = time.time()
for v in sample_nodes:
    lengths = nx.single_source_shortest_path_length(G,v)
    eccentricities_sample[v] = max(lengths.values())

diameter_est = max(eccentricities_sample.values())
radius_est = min(eccentricities_sample.values())
elapsed = time.time() - t0
print(f"  Diâmetro estimado : {diameter_est}  (SNAP reporta 17)")
print(f"  Raio estimado     : {radius_est}")
print(f"  Tempo amostragem  : {elapsed:.1f}s  (k={SAMPLE_K})")
print("  ATENÇÃO: valores exatos inviáveis para grafos desta escala.")
print("           Usar estimativa por amostragem + valor SNAP.")


[5/9] Diâmetro e Raio (estimativa por amostragem)...
  Diâmetro estimado : 26  (SNAP reporta 17)
  Raio estimado     : 17
  Tempo amostragem  : 242.6s  (k=500)
  ATENÇÃO: valores exatos inviáveis para grafos desta escala.
           Usar estimativa por amostragem + valor SNAP.


## 6. Comprimento medio dos caminhos

In [28]:
print("\n[6/9] Comprimento médio dos caminhos (estimativa)...")
 
total_path = 0
count_path = 0
for v in sample_nodes:
    lengths = nx.single_source_shortest_path_length(G, v)
    total_path += sum(lengths.values()) - 0  # inclui distância 0 para si
    count_path += len(lengths) - 1           # exclui self
 
avg_path_est = total_path / count_path if count_path > 0 else float("inf")
print(f"  Comprimento médio estimado: {avg_path_est:.4f}  (SNAP: ~6.5 para 90-percentil)")
print("  ATENÇÃO: estimativa por amostragem BFS em {SAMPLE_K} nós.")


[6/9] Comprimento médio dos caminhos (estimativa)...
  Comprimento médio estimado: 8.9198  (SNAP: ~6.5 para 90-percentil)
  ATENÇÃO: estimativa por amostragem BFS em {SAMPLE_K} nós.


## 7. Coeficiente de clusteriação medio 

In [29]:
print("\n[7/9] Coeficiente de clusterização...")


[7/9] Coeficiente de clusterização...


In [30]:
avg_clustering = nx.average_clustering(G, nodes=sample_nodes)
print(f"  Coef. clusterização médio (amostrado): {avg_clustering:.4f}  (SNAP: 0.2843)")
print("  ATENÇÃO: estimativa por amostragem.")

  Coef. clusterização médio (amostrado): 0.0237  (SNAP: 0.2843)
  ATENÇÃO: estimativa por amostragem.


## 8. Número de triângulos 

In [31]:
USE_SNAP_TRIANGLES = True

if USE_SNAP_TRIANGLES:
    print("  Calculando triângulos... ")
    t0 = time.time()
    tri = nx.triangles(G)
    n_triangles = sum(tri.values()) // 3
    print(f"  Triângulos: {n_triangles:,}  ({time.time()-t0:.1f}s)") 
    print("  Calculando triângulos... (pode demorar)")
    t0 = time.time()
    tri = nx.triangles(G)
    n_triangles = sum(tri.values()) // 3
    print(f"  Triângulos: {n_triangles:,}  ({time.time()-t0:.1f}s)")

  Calculando triângulos... 
  Triângulos: 16,886  (1.1s)
  Calculando triângulos... (pode demorar)
  Triângulos: 16,886  (1.1s)


## 9. Resumo 

In [ ]:
print("\n" + "=" * 60)
print("TABELA RESUMO — Parte I")
print("=" * 60)
 
rows = [
    ("Número de vértices",               f"{n_nodes:,}",                    "Exato"),
    ("Número de arestas",                f"{n_edges:,}",                    "Exato"),
    ("Grau mínimo",                      str(deg_min),                      "Exato"),
    ("Grau máximo",                      str(deg_max),                      "Exato"),
    ("Grau médio",                       f"{deg_avg:.4f}",                  "Exato"),
    ("Distribuição de graus",            "Ver figura (log-log)",            "Exato"),
    ("Densidade",                        f"{density:.8f}",                  "Exato"),
    ("Nº componentes conexas",           str(n_comp),                       "Exato"),
    ("Tamanho das componentes",          str(comp_sizes[:3]) + "...",       "Exato"),
    ("Diâmetro",                         f"~{diameter_est} (SNAP: 17)",    "Estimativa"),
    ("Raio",                             f"~{radius_est}",                  "Estimativa"),
    ("Comprimento médio dos caminhos",   f"~{avg_path_est:.2f}",           "Estimativa"),
    ("Coef. clusterização médio",        f"~{avg_clustering:.4f}",         "Estimativa"),
    ("Número de triângulos",             f"{n_triangles:,}",               "SNAP / Exato"),
]
 
print(f"{'Métrica':<40} {'Valor':<28} {'Tipo'}")
print("-" * 80)
for m, v, t in rows:
    print(f"{m:<40} {v:<28} {t}")
 
print("\nNota: métricas marcadas como 'Estimativa' foram computadas por")
print("amostragem BFS (k=500 nós) devido à escala do grafo (~4M nós).")


TABELA RESUMO — Parte I
Métrica                                  Valor                        Tipo
--------------------------------------------------------------------------------
Número de vértices                       427,701                      Exato
Número de arestas                        303,526                      Exato
Grau mínimo                              1                            Exato
Grau máximo                              221                          Exato
Grau médio                               2.8182                       Exato
Distribuição de graus                    Ver figura (log-log)         Exato
Densidade                                0.00000928                   Exato
Nº componentes conexas                   1                            Exato
Tamanho das componentes                  [303526]...                  Exato
Diâmetro                                 ~26 (SNAP: 17)               Estimativa
Raio                                     ~17          